In [ ]:
# Example 1: Instantiate the URLLib3Instrumentor and inspect its public API.
from opentelemetry.instrumentation.urllib3 import URLLib3Instrumentor

instrumentor = URLLib3Instrumentor()
print(f"type = {type(instrumentor).__name__}")
print(f"mro  = {[c.__name__ for c in type(instrumentor).__mro__]}")

# BaseInstrumentor exposes these methods.
for attr in ('instrument', 'uninstrument', 'instrumentation_dependencies'):
    assert hasattr(instrumentor, attr), f"missing {attr}"
    print(f"  has {attr}: {callable(getattr(instrumentor, attr))}")

deps = instrumentor.instrumentation_dependencies()
print(f"instrumentation_dependencies = {list(deps)}")
assert any('urllib3' in d for d in deps), "expected urllib3 in dependency list"


In [ ]:
# Example 2: instrument() + uninstrument() round-trip should be idempotent.
from opentelemetry.instrumentation.urllib3 import URLLib3Instrumentor

instrumentor = URLLib3Instrumentor()

# instrument is safe to call; subsequent calls are no-ops while already instrumented.
instrumentor.instrument()
try:
    # second call: BaseInstrumentor swallows the duplicate instrument call.
    instrumentor.instrument()
    print("instrument() x2 OK")
finally:
    instrumentor.uninstrument()
    # uninstrument again — also a no-op.
    instrumentor.uninstrument()
    print("uninstrument() x2 OK")


In [ ]:
# Example 3: End-to-end — make a urllib3 HTTP request against a local in-process
# server with instrumentation enabled, and verify a span is produced (offline).
import http.server
import threading
import urllib3
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from opentelemetry.instrumentation.urllib3 import URLLib3Instrumentor

class _Handler(http.server.BaseHTTPRequestHandler):
    def do_GET(self):
        self.send_response(200); self.send_header("Content-Length", "2"); self.end_headers()
        self.wfile.write(b"ok")
    def log_message(self, *args): pass

server = http.server.HTTPServer(("127.0.0.1", 0), _Handler)
threading.Thread(target=server.serve_forever, daemon=True).start()
exporter = InMemorySpanExporter()
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(exporter))

URLLib3Instrumentor().instrument(tracer_provider=provider)
try:
    resp = urllib3.PoolManager().request(
        "GET", f"http://127.0.0.1:{server.server_address[1]}/", timeout=10.0)
    print(f"status = {resp.status}")
finally:
    URLLib3Instrumentor().uninstrument()
    server.shutdown()

spans = exporter.get_finished_spans()
print(f"captured {len(spans)} span(s); first name = {spans[0].name if spans else None}")
assert len(spans) >= 1, "expected at least one span from urllib3 request"

In [ ]:
# Example 4: The instrumented span carries HTTP semantic-convention attributes
# (method + url + status), demonstrated against a local in-process server.
import http.server
import threading
import urllib3
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from opentelemetry.instrumentation.urllib3 import URLLib3Instrumentor

class _Handler(http.server.BaseHTTPRequestHandler):
    def do_GET(self):
        self.send_response(204); self.send_header("Content-Length", "0"); self.end_headers()
    def log_message(self, *args): pass

server = http.server.HTTPServer(("127.0.0.1", 0), _Handler)
threading.Thread(target=server.serve_forever, daemon=True).start()
exporter = InMemorySpanExporter()
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(exporter))

URLLib3Instrumentor().instrument(tracer_provider=provider)
try:
    urllib3.PoolManager().request(
        "GET", f"http://127.0.0.1:{server.server_address[1]}/status/204", timeout=10.0)
finally:
    URLLib3Instrumentor().uninstrument()
    server.shutdown()

spans = exporter.get_finished_spans()
attrs = dict(spans[0].attributes) if spans else {}
print(f"span attributes (subset) = {[k for k in attrs if 'http' in k.lower() or 'url' in k.lower()][:8]}")
assert spans, "expected a span"
assert any("http" in k.lower() or "url" in k.lower() for k in attrs)

In [ ]:
# Example 5: request_hook + response_hook are user-provided callbacks invoked
# during the instrumented request lifecycle. Verify both fire (against a local server).
# Current hook signatures: request_hook(span, pool, request_info) /
# response_hook(span, pool, response).
import http.server
import threading
import urllib3
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from opentelemetry.instrumentation.urllib3 import URLLib3Instrumentor

class _Handler(http.server.BaseHTTPRequestHandler):
    def do_GET(self):
        self.send_response(200); self.send_header("Content-Length", "2"); self.end_headers()
        self.wfile.write(b"ok")
    def log_message(self, *args): pass

server = http.server.HTTPServer(("127.0.0.1", 0), _Handler)
threading.Thread(target=server.serve_forever, daemon=True).start()
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(InMemorySpanExporter()))

hook_log = []
def request_hook(span, pool, request_info):
    hook_log.append(("request", type(request_info).__name__))
def response_hook(span, pool, response):
    hook_log.append(("response", getattr(response, "status", None)))

URLLib3Instrumentor().instrument(
    tracer_provider=provider, request_hook=request_hook, response_hook=response_hook)
try:
    urllib3.PoolManager().request(
        "GET", f"http://127.0.0.1:{server.server_address[1]}/", timeout=10.0)
finally:
    URLLib3Instrumentor().uninstrument()
    server.shutdown()

print(f"hook_log = {hook_log}")
assert any(e[0] == "request" for e in hook_log), "request_hook never fired"
assert any(e[0] == "response" for e in hook_log), "response_hook never fired"